In [3]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
import gc

In [4]:
# 1) Аккуратно остановим старый контекст/сессию (если висят)
try:
    if SparkContext._active_spark_context is not None:
        SparkContext._active_spark_context.stop()
except Exception as e:
    print("Stop old SparkContext:", e)

try:
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
except Exception as e:
    print("Stop old SparkSession:", e)

gc.collect()  # иногда помогает JVM освободить ресурсы

# 2) Поднимем новую локальную сессию
spark = (
    SparkSession.builder
        .appName("Text session")
        .master("local[4]")
        # в local-режиме важна именно память драйвера:
        .config("spark.driver.memory", "4g")
        # executor'ов как таковых нет, но оставить не мешает:
        .config("spark.executor.memory", "2g")
        # КЛЮЧЕВОЕ:
        .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
        .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
        .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 15:47:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.0


In [5]:
spark.stop()

In [3]:
!java -version 

openjdk version "17.0.15" 2025-04-15
OpenJDK Runtime Environment Homebrew (build 17.0.15+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.15+0, mixed mode, sharing)


In [1]:
import os

In [2]:
os.environ.get("JAVA_HOME")

'/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'

export JAVA_HOME="/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

export PATH=$JAVA_HOME/bin:$PATH